# 🫁 Lung Cancer Detection — Explainable Vision Transformer (ViT)
**3-Class Classification:** Lung Benign · Lung Adenocarcinoma (ACA) · Lung Squamous Cell Carcinoma (SCC)

---
### Pipeline (Paper-Ready):
1. GPU check & library install
2. Mount Drive & verify dataset
3. Imports, config & transforms
4. Data splitting + SubsetWithTransform
5. DataLoaders + class weights
6. ViT-Base/16 (3-class head)
7. Weighted loss / AdamW / cosine scheduler
8. Training loop
9. **Figure 4** — Training curves  → `paper_figures/fig4_lung_training_curves.png`
10. **Figure 5** — Confusion matrix → `paper_figures/fig5_lung_confusion_matrix.png`
11. AUC-ROC (one-vs-rest)          → `paper_figures/lung_roc_curves.png`
12. **Figure 6b** — XAI dual panel  → `paper_figures/fig6b_lung_xai_rollout_gradcam.png`
13. Multi-modal fusion blueprint
14. Multi-organ extension config
15. Save model & download all figures
---


## ✅ Step 0 — GPU Check


In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)


## 📦 Step 1 — Install Libraries


In [ ]:
!pip install -q timm transformers scikit-learn matplotlib seaborn


## 📁 Step 2 — Mount Drive & Verify Dataset
```
dataset2/
├── lung_n/      ← benign
├── lung_aca/    ← adenocarcinoma
└── lung_scc/    ← squamous cell carcinoma
```


In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/dataset2'   # ← change if needed
print('📂 Dataset contents:')
for cls in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, cls)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png','.tif','.tiff'))])
        print(f'   {cls}: {n:,} images')


## 🔧 Step 3 — Imports & Config


In [ ]:
import torch, os, warnings
import torch.nn as nn, torch.optim as optim, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
import timm, numpy as np, matplotlib, seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from collections import Counter
from PIL import Image
import matplotlib.pyplot as plt
matplotlib.rcParams['figure.dpi'] = 150
warnings.filterwarnings('ignore')

DATA_DIR    = '/content/drive/MyDrive/dataset2'
IMG_SIZE    = 224
BATCH_SIZE  = 32
NUM_EPOCHS  = 15
LR          = 2e-4
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
SEED        = 42
NUM_CLASSES = 3
FIGURES_DIR = 'paper_figures'
os.makedirs(FIGURES_DIR, exist_ok=True)
torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Config ready —', device)
print(f'   Figures → {FIGURES_DIR}/')


## 🖼️ Step 4 — Transforms, Dataset & Split


In [ ]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

full_dataset = datasets.ImageFolder(root=DATA_DIR, transform=train_tf)
class_names  = full_dataset.classes
num_classes  = len(class_names)
label_counts = Counter(full_dataset.targets)
print('Classes:', class_names)
print('Total images:', len(full_dataset))
for i, n in enumerate(class_names): print(f'   [{i}] {n}: {label_counts[i]:,}')

n_total = len(full_dataset)
n_train = int(n_total * TRAIN_RATIO)
n_val   = int(n_total * VAL_RATIO)
n_test  = n_total - n_train - n_val
train_set, val_set, test_set = random_split(
    full_dataset, [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(SEED))

class SubsetWithTransform(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset; self.transform = transform
    def __len__(self): return len(self.subset)
    def __getitem__(self, idx):
        path, label = self.subset.dataset.imgs[self.subset.indices[idx]]
        return self.transform(Image.open(path).convert('RGB')), label

val_set_clean  = SubsetWithTransform(val_set,  val_tf)
test_set_clean = SubsetWithTransform(test_set, val_tf)
print(f'Split — Train:{n_train:,} Val:{n_val:,} Test:{n_test:,}')


## 🚀 Step 5 — DataLoaders & Class Weights


In [ ]:
class_weights = torch.tensor(
    [n_train / (num_classes * label_counts[i]) for i in range(num_classes)], dtype=torch.float
).to(device)
print('Class weights:', class_weights.cpu().numpy().round(4))
train_loader = DataLoader(train_set,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set_clean,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set_clean, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print('✅ DataLoaders ready')


## 🤖 Step 6 — ViT-Base/16 (3-Class Head)


In [ ]:
model = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=num_classes)
model = model.to(device)
total_p = sum(p.numel() for p in model.parameters())
print(f'Head: {model.head}')
print(f'Total parameters: {total_p:,}')


## ⚙️ Step 7 — Weighted Loss / AdamW / Cosine Scheduler


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
print('✅ Loss / Optimizer / Scheduler ready')


## 🏋️ Step 8 — Training Loop


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train(); rl, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        rl += loss.item() * imgs.size(0)
        correct += (out.argmax(1) == labels).sum().item(); total += labels.size(0)
    return rl/total, correct/total

def evaluate(model, loader, criterion, device):
    model.eval(); rl, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out = model(imgs); loss = criterion(out, labels)
            rl += loss.item()*imgs.size(0)
            correct += (out.argmax(1)==labels).sum().item(); total += labels.size(0)
    return rl/total, correct/total

history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
best_val_acc = 0.0; best_model_path = 'best_lung_vit.pth'
print(f'Training {NUM_EPOCHS} epochs on {device}...'); print('='*70)
for epoch in range(1, NUM_EPOCHS+1):
    tl, ta = train_one_epoch(model, train_loader, optimizer, criterion, device)
    vl, va = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    for k, v in zip(['train_loss','val_loss','train_acc','val_acc'],[tl,vl,ta,va]):
        history[k].append(v)
    saved = ''
    if va > best_val_acc:
        best_val_acc = va; torch.save(model.state_dict(), best_model_path); saved = '  ✅ saved'
    print(f'Ep {epoch:02d}/{NUM_EPOCHS} | Train L:{tl:.4f} A:{ta:.4f} | Val L:{vl:.4f} A:{va:.4f}{saved}')
print('='*70); print(f'Best Val Acc: {best_val_acc:.4f}')


## 📈 Step 9 — Figure 4: Training Curves (Paper Figure)
> `paper_figures/fig4_lung_training_curves.png` → LaTeX Figure 4


In [ ]:
er = range(1, NUM_EPOCHS+1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle('Lung Cancer ViT — Training Curves (3-Class)', fontsize=13, fontweight='bold', y=1.02)
ax1.plot(er, history['train_loss'], 'b-o', lw=2, ms=5, label='Train Loss')
ax1.plot(er, history['val_loss'],   'r-o', lw=2, ms=5, label='Val Loss')
ax1.set_title('Loss per Epoch', fontweight='bold'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3); ax1.set_xticks(list(er))
ax2.plot(er, [a*100 for a in history['train_acc']], 'b-o', lw=2, ms=5, label='Train Acc')
ax2.plot(er, [a*100 for a in history['val_acc']],   'r-o', lw=2, ms=5, label='Val Acc')
ax2.set_title('Accuracy per Epoch (%)', fontweight='bold'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_ylim([50, 102]); ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_xticks(list(er))
plt.tight_layout()
out = f'{FIGURES_DIR}/fig4_lung_training_curves.png'
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show(); print(f'✅ Saved: {out}')


## 🧪 Step 10 — Test Evaluation


In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        out   = model(imgs.to(device))
        probs = torch.softmax(out, dim=1).cpu()
        all_preds.extend(probs.argmax(dim=1).numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.numpy())
all_preds = np.array(all_preds); all_labels = np.array(all_labels); all_probs = np.array(all_probs)
test_acc  = (all_preds == all_labels).mean()
print(f'\n🎯 Test Accuracy: {test_acc*100:.2f}%')
print(classification_report(all_labels, all_preds, target_names=class_names))


## 📊 Step 11 — Figure 5: Confusion Matrix (Paper Figure)
> `paper_figures/fig5_lung_confusion_matrix.png` → LaTeX Figure 5


In [ ]:
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names,
            linewidths=0.8, annot_kws={'size':13,'weight':'bold'}, ax=ax)
ax.set_title(f'Confusion Matrix — Lung Cancer Test Set (3-Class)\n(n={len(all_labels):,})',
             fontweight='bold', fontsize=12, pad=12)
ax.set_ylabel('True Label', fontsize=11); ax.set_xlabel('Predicted Label', fontsize=11)
plt.tight_layout()
out = f'{FIGURES_DIR}/fig5_lung_confusion_matrix.png'
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show(); print(f'✅ Saved: {out}')


## 📉 Step 12 — AUC-ROC One-vs-Rest
> `paper_figures/lung_roc_curves.png`


In [ ]:
y_bin = label_binarize(all_labels, classes=list(range(num_classes)))
colors_roc = ['#e41a1c','#377eb8','#4daf4a']
fig, ax = plt.subplots(figsize=(6, 5.5))
for i, (cls, c) in enumerate(zip(class_names, colors_roc)):
    fpr, tpr, _ = roc_curve(y_bin[:,i], all_probs[:,i])
    ax.plot(fpr, tpr, color=c, lw=2, label=f'{cls} (AUC={auc(fpr,tpr):.4f})')
ax.plot([0,1],[0,1],'k--',lw=1.2,label='Random')
ax.set(xlim=[0,1],ylim=[0,1.02],xlabel='FPR',ylabel='TPR',
       title='ROC Curves — Lung Cancer (One-vs-Rest)')
ax.set_xlabel('False Positive Rate', fontsize=11); ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — Lung Cancer (One-vs-Rest)', fontweight='bold')
ax.legend(loc='lower right',fontsize=9.5); ax.grid(True,alpha=0.3)
plt.tight_layout()
out = f'{FIGURES_DIR}/lung_roc_curves.png'
plt.savefig(out, dpi=300, bbox_inches='tight'); plt.show(); print(f'✅ Saved: {out}')


## 🔍 Step 13 — XAI Functions: Attention Rollout + Grad-CAM


In [ ]:
# ── Attention Rollout ─────────────────────────────────────────────
def get_attention_rollout(model, img_tensor, device, head_fusion='mean', discard_ratio=0.9):
    model.eval(); attention_maps = []
    def hook_fn(module, inp, out):
        B, N, C = inp[0].shape
        qkv = module.qkv(inp[0]).reshape(B,N,3,module.num_heads,C//module.num_heads).permute(2,0,3,1,4)
        q, k, _ = qkv.unbind(0)
        attn = (q @ k.transpose(-2,-1)) * (C//module.num_heads)**-0.5
        attention_maps.append(attn.softmax(dim=-1).detach().cpu())
    hooks = [block.attn.register_forward_hook(hook_fn) for block in model.blocks]
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
        pred_class = out.argmax(1).item()
        confidence = torch.softmax(out,dim=1).max().item()
    for h in hooks: h.remove()
    rollout = torch.eye(attention_maps[0].shape[-1])
    for attn in attention_maps:
        a = {'mean':attn.mean,'min':attn.min,'max':attn.max}[head_fusion](dim=1)
        if hasattr(a,'values'): a = a.values
        a = a[0]
        flat = a.view(-1); thresh = torch.quantile(flat, discard_ratio)
        a[a < thresh] = 0
        a = a + torch.eye(a.shape[0]); a /= a.sum(dim=-1,keepdim=True)
        rollout = a @ rollout
    mask = rollout[0,1:]; gs = int(mask.shape[0]**0.5)
    mask = mask.reshape(gs,gs).numpy()
    return (mask-mask.min())/(mask.max()-mask.min()+1e-8), pred_class, confidence

# ── Grad-CAM for ViT ──────────────────────────────────────────────
class ViTGradCAM:
    def __init__(self, model, target_layer=None):
        self.model=model; self.activations=None; self.gradients=None
        target = target_layer or model.blocks[-1]
        self._fh = target.register_forward_hook(lambda m,i,o: setattr(self,'activations',o.detach()))
        self._bh = target.register_full_backward_hook(lambda m,gi,go: setattr(self,'gradients',go[0].detach()))
    def generate(self, img_tensor, device, class_idx=None):
        self.model.eval()
        img_t = img_tensor.unsqueeze(0).to(device); img_t.requires_grad_(True)
        out = self.model(img_t)
        pred_class = out.argmax(1).item(); confidence = torch.softmax(out,dim=1).max().item()
        self.model.zero_grad(); out[0, class_idx if class_idx is not None else pred_class].backward()
        weights = self.gradients[0,1:].mean(dim=-1)
        cam = F.relu((weights.unsqueeze(-1)*self.activations[0,1:]).sum(dim=-1))
        gs = int(cam.shape[0]**0.5); cam = cam.reshape(gs,gs).cpu().numpy()
        cam = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
        cam = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR))/255.0
        return cam, pred_class, confidence
    def remove_hooks(self): self._fh.remove(); self._bh.remove()

print('✅ XAI functions ready (Attention Rollout + ViTGradCAM)')


## 🔬 Step 14 — Figure 6b: XAI Dual Comparison (Paper Figure)
> `paper_figures/fig6b_lung_xai_rollout_gradcam.png` → LaTeX Figure 6


In [ ]:
def plot_dual_xai(model, dataset, class_names, device, num_samples=6, save_path=None):
    gradcam = ViTGradCAM(model)
    mean_np = np.array([0.485,0.456,0.406]); std_np = np.array([0.229,0.224,0.225])
    np.random.seed(42)
    idx_by_cls = {i:[] for i in range(len(class_names))}
    for idx in range(len(dataset)):
        _, lbl = dataset[idx]; idx_by_cls[lbl].append(idx)
    spc = num_samples // len(class_names)
    chosen = []
    for ci in range(len(class_names)):
        chosen.extend(np.random.choice(idx_by_cls[ci], min(spc,len(idx_by_cls[ci])), replace=False))
    indices = chosen[:num_samples]

    fig, axes = plt.subplots(num_samples, 3, figsize=(13, num_samples*3.2))
    fig.suptitle('XAI — Attention Rollout vs Grad-CAM (Lung Cancer ViT)', fontsize=13, fontweight='bold', y=1.01)
    for col, title in enumerate(['Original Image','Attention Rollout Overlay','Grad-CAM Overlay']):
        axes[0,col].set_title(title, fontsize=11, fontweight='bold', pad=8)

    for row, idx in enumerate(indices):
        img_t, true_lbl = dataset[idx]
        img_np = np.clip(img_t.permute(1,2,0).numpy()*std_np+mean_np, 0, 1)
        ar_mask,pred_ar,conf_ar = get_attention_rollout(model, img_t, device)
        ar_up = np.array(Image.fromarray((ar_mask*255).astype(np.uint8)).resize((IMG_SIZE,IMG_SIZE),Image.BILINEAR))/255.0
        gc_mask,pred_gc,conf_gc = gradcam.generate(img_t, device)
        ok = pred_gc==true_lbl; color = '#1a7a1a' if ok else '#cc0000'; mark = '✓' if ok else '✗'

        axes[row,0].imshow(img_np)
        axes[row,0].set_ylabel(f'True: {class_names[true_lbl]}', fontsize=9, labelpad=4)
        axes[row,0].tick_params(left=False,bottom=False,labelleft=False,labelbottom=False)
        for sp in axes[row,0].spines.values(): sp.set_visible(False)

        axes[row,1].imshow(img_np); axes[row,1].imshow(ar_up, alpha=0.45, cmap='jet')
        axes[row,1].set_title(f'AR Pred:{class_names[pred_ar]} ({conf_ar*100:.1f}%)', fontsize=9, color=color)
        axes[row,1].axis('off')

        axes[row,2].imshow(img_np); im=axes[row,2].imshow(gc_mask, alpha=0.45, cmap='jet', vmin=0, vmax=1)
        axes[row,2].set_title(f'{mark} GC Pred:{class_names[pred_gc]} ({conf_gc*100:.1f}%)', fontsize=9, color=color)
        axes[row,2].axis('off')

    cbar_ax = fig.add_axes([0.92,0.15,0.015,0.7])
    fig.colorbar(im, cax=cbar_ax, label='Activation intensity')
    plt.tight_layout(rect=[0,0,0.91,1])
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight'); print(f'✅ Saved: {save_path}')
    plt.show(); gradcam.remove_hooks()

plot_dual_xai(model, test_set_clean, class_names, device, num_samples=6,
              save_path=f'{FIGURES_DIR}/fig6b_lung_xai_rollout_gradcam.png')


## 🏗️ Step 15 — Multi-Modal Fusion Blueprint


In [ ]:
class MultiModalCancerClassifier(nn.Module):
    """Fuses histopathology + radiology + clinical metadata for cancer classification."""
    def __init__(self, num_classes=3, meta_dim=8, fusion_dim=512, dropout=0.3):
        super().__init__()
        self.histo = timm.create_model('vit_base_patch16_224',  pretrained=True, num_classes=0)  # 768
        self.radio = timm.create_model('vit_small_patch16_224', pretrained=True, num_classes=0)  # 384
        self.meta  = nn.Sequential(nn.Linear(meta_dim,64),nn.LayerNorm(64),nn.GELU(),nn.Linear(64,128),nn.GELU())
        self.head  = nn.Sequential(
            nn.Linear(1280,fusion_dim),nn.LayerNorm(fusion_dim),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(fusion_dim,fusion_dim//2),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(fusion_dim//2,num_classes))
    def forward(self, h, r, m):
        return self.head(torch.cat([self.histo(h), self.radio(r), self.meta(m)], dim=1))

mm = MultiModalCancerClassifier(num_classes=3)
with torch.no_grad():
    out = mm(torch.randn(2,3,224,224), torch.randn(2,3,224,224), torch.randn(2,8))
print('MultiModal output shape:', out.shape, '← (batch=2, 3 classes)')
print(f'Total parameters: {sum(p.numel() for p in mm.parameters()):,}')


## 🔄 Step 16 — Multi-Organ Extension Config


In [ ]:
MULTI_ORGAN_CONFIG = {
    'lung'  : {'classes':['lung_n','lung_aca','lung_scc'],      'num_classes':3},
    'colon' : {'classes':['colon_aca','colon_n'],               'num_classes':2},
    'breast': {'classes':['breast_benign','breast_malignant'],  'num_classes':2},
    'ovary' : {'classes':['ovary_benign','ovary_malignant'],    'num_classes':2},
}
def build_model_for_organ(organ):
    cfg = MULTI_ORGAN_CONFIG[organ]
    m   = timm.create_model('vit_base_patch16_224',pretrained=True,num_classes=cfg['num_classes'])
    print(f'[{organ}] head: {m.head}  classes: {cfg["classes"]}')
    return m, cfg['classes']
print('Available organs:', list(MULTI_ORGAN_CONFIG.keys()))


## 💾 Step 17 — Save Model & Download All Paper Figures


In [ ]:
torch.save({'model_state_dict':model.state_dict(),'class_names':class_names,
             'best_val_accuracy':best_val_acc,'test_accuracy':test_acc,
             'config':{'img_size':IMG_SIZE,'batch_size':BATCH_SIZE,
                       'num_epochs':NUM_EPOCHS,'lr':LR,'num_classes':num_classes,'task':'lung_3class'}},
            'lung_cancer_vit_final.pth')
print('✅ Model saved')

from google.colab import files
all_outputs = ['lung_cancer_vit_final.pth'] + [
    os.path.join(FIGURES_DIR,f) for f in sorted(os.listdir(FIGURES_DIR))]
print('\n📦 Downloading:')
for f in all_outputs:
    if os.path.exists(f):
        print(f'   {f}  ({os.path.getsize(f)/1024:.1f} KB)')
        files.download(f)
print('\n✅ Done.')


## 🔮 Step 18 — Single Image Inference (Dual XAI)


In [ ]:
def predict_with_dual_xai(model, image_path, class_names, device):
    tf = transforms.Compose([transforms.Resize((224,224)),transforms.ToTensor(),
                              transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    pil = Image.open(image_path).convert('RGB'); t = tf(pil)
    ar_mask,pred_ar,conf_ar = get_attention_rollout(model,t,device)
    ar_up = np.array(Image.fromarray((ar_mask*255).astype(np.uint8)).resize((224,224),Image.BILINEAR))/255.0
    gc = ViTGradCAM(model); gc_mask,pred_gc,conf_gc = gc.generate(t,device); gc.remove_hooks()
    model.eval()
    with torch.no_grad():
        probs = torch.softmax(model(t.unsqueeze(0).to(device)),dim=1)[0].cpu().numpy()
    print(f'Prediction: {class_names[pred_gc]} ({conf_gc*100:.1f}%)')
    for i,n in enumerate(class_names): print(f'   {n:20s} {probs[i]*100:5.1f}%  {"█"*int(probs[i]*40)}')
    img_np = np.array(pil.resize((224,224)))/255.0
    fig,axes = plt.subplots(1,3,figsize=(13,4))
    axes[0].imshow(img_np); axes[0].set_title('Original',fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(img_np); axes[1].imshow(ar_up,alpha=0.45,cmap='jet')
    axes[1].set_title(f'Rollout: {class_names[pred_ar]} ({conf_ar*100:.1f}%)',fontweight='bold'); axes[1].axis('off')
    axes[2].imshow(img_np); axes[2].imshow(gc_mask,alpha=0.45,cmap='jet')
    axes[2].set_title(f'Grad-CAM: {class_names[pred_gc]} ({conf_gc*100:.1f}%)',fontweight='bold'); axes[2].axis('off')
    plt.tight_layout(); plt.show()

# from google.colab import files
# uploaded = files.upload()
# predict_with_dual_xai(model, list(uploaded.keys())[0], class_names, device)
print('✅ Ready — uncomment above to test a single image')
